# Day 90: NVIDIA Dynamo Disaggregated Serving

**Layer:** Advanced


## Concept Overview

NVIDIA Dynamo disaggregates prefill and decode into separate worker pools. Prefill workers process long prompt contexts on compute-optimized nodes; decode workers run autoregressive generation on memory-bandwidth-optimized nodes.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Disaggregated Serving Model

Model the roofline for prefill (compute-bound) vs decode (memory-bound) and show why separate worker pools improve utilization.


In [ ]:
import numpy as np, matplotlib.pyplot as plt

def roofline(flops, bytes_moved, peak_tflops=67, peak_bw_tbs=0.273):
    compute_time = flops / (peak_tflops * 1e12)
    memory_time = bytes_moved / (peak_bw_tbs * 1e12)
    bound = "compute" if compute_time > memory_time else "memory"
    return max(compute_time, memory_time) * 1000, bound

# 8B model, d=4096, 32 heads
d, n_heads, n_layers = 4096, 32, 32
bytes_weights = d*d*4*n_layers * 2  # 4 matrices, FP16

# Prefill: B=1, S=2048
S = 2048; B = 1
flops_prefill = 2 * B * S * d * 4 * d * n_layers
bytes_prefill = bytes_weights + B*S*d*2*n_layers  # activations

# Decode: B=1, single token
flops_decode = 2 * B * 1 * d * 4 * d * n_layers
bytes_decode = bytes_weights  # load all weights for each token

print("Roofline analysis (DGX Spark):")
for name, flops, bts in [("Prefill S=2048",flops_prefill,bytes_prefill),
                          ("Decode 1 token",flops_decode,bytes_decode)]:
    t, bound = roofline(flops, bts)
    arith_intens = flops / bts
    print(f"  {name:<20}: {t:.2f}ms ({bound}-bound) AI={arith_intens:.1f}")

print()
print("Disaggregation benefit: decode workers never wait for prefill computation.")
print("Prefill workers: use large batches to stay compute-bound.")
print("Decode workers: use large KV caches to maximize memory bandwidth.")


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- NVIDIA Dynamo disaggregates prefill and decode into separate worker pools.
- NVIDIA Dynamo disaggregates prefill and decode into separate worker pools. Prefi....
- Day 90 implementation complete.
